# Graph Games with Reinforcement Learning

This notebook explores **graph construction as a reinforcement learning problem**. The goal is to build a graph — by toggling edges — so that the resulting graph optimises some graph-theoretic objective, such as maximising the diameter or achieving a regular graph.

Two different approaches are considered:
- a **single-player game**, in which one agent is trained;
- a **two-player game**, in which two agents cooperate to achieve the same goal.

The project uses:
- **`gymnasium`** – standard RL environment interface
- **`stable-baselines3-contrib` (`MaskablePPO`)** – a variant of Proximal Policy Optimisation (PPO) that supports action masking, which is essential here because not all edge-toggle moves are legal at every step
- **`networkx`** – graph analysis (diameter, connectivity, degree sequences)

---

In [ ]:
import numpy as np
import gymnasium as gym
import networkx as nx
from sb3_contrib import MaskablePPO
from sb3_contrib.common.wrappers import ActionMasker
import matplotlib.pyplot as plt
import tempfile
import webbrowser
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML, display
from tqdm import tqdm

    

---
# Single-player game

This first section develops a **one-player environment**. A single agent controls the entire graph-building process by toggling edges one at a time. The goal here is simply to verify that the environment and the reward signal work before adding a second agent.

### `GraphWorldWithUndo` 
This is the core `gymnasium.Env` subclass for the single-agent setting.

**State / observation space**  
The graph is represented as a symmetric `nverts × nverts` adjacency matrix (entries 0 or 1). This is what the agent observes at each step.

**Action space**  
There are `nverts*(nverts-1)/2 + 1` discrete actions:
- Actions `0 … N-2` each correspond to a unique edge `(i, j)` via the triangular indexing in `_get_edge`.
- The **last action** is a *resign / terminate* signal — the agent can stop at any time.

**Edge toggle ("undo") mechanism**  
When the agent picks an edge `(i, j)`, the code does:
```python
matrix[i,j] += 1
matrix[i,j] %= 2
```
This is a modulo-2 toggle: if the edge exists it is removed, and if it is absent it is added. This gives the agent full flexibility to both add and remove edges in a single, unified action space.



**Reward (`_reward`)**  
To guide the reinforcement learning agent, the reward is based on how the graph diameter changes after each move. Notice that we do not assume the graph to be connected during training. A constant penalty applied after each step (-0.1 in this case) discourages inefficient behaviours such as repeatedly adding and removing the same edge (so we avoid to fall into loops).


In [ ]:
# Our base model
class GraphWorldWithUndo(gym.Env):
    def __init__(self, nverts):
        self.nverts = nverts
    
        # Adjacency matrix with column describing move
        self.observation_space = gym.spaces.Box(
            0, 1, shape=(nverts, nverts), dtype=np.int8
        )

        self._n_actions = nverts*(nverts-1)//2 +1 
        self._steps = 0

        self.action_space = gym.spaces.Discrete(self._n_actions)
        self.render_mode = "ansi"
        self.previous_invariant = 0
        self.reset()
        
   

    def _get_edge(self, action):
        if action == self._n_actions - 1:
            return None  # terminate
    
        count = 0
        for i in range(self.nverts):
            for j in range(i+1, self.nverts):
                if count == action:
                    return i, j
                count += 1
    


    def _legal_actions(self):
        mask = np.ones(self._n_actions, dtype=bool)
        return mask
    
    def _reward(self):
        adj_mat = self._matrix[:]
        G = nx.from_numpy_array(adj_mat)
        CC = list(nx.connected_components(G))
        d = max(nx.diameter(nx.subgraph(G, comp)) for comp in CC)
        reward = d - self.previousdiameter - 0.1
        self.previousdiameter = d
        return reward
       
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self._matrix = np.zeros((self.nverts,self.nverts),dtype=np.int8)
        self.previousdiameter = 0
        self.previous_invariant = self._reward()
        self._steps = 0
        info = {"action_mask": self._legal_actions()}
        return self._matrix, info
        

    def step(self, action):
        self._steps += 1
        info = {"action_mask": self._legal_actions()}
        # Resignation
        if action == self.nverts-1:
            terminated = True
            return self._matrix, 0, terminated, False, info
        # Otherwise we play :)

        terminated = False
        if action == self._n_actions-1:
            terminated = True
            return self._matrix, 0, terminated, False, info
        
    
        edge = self._get_edge(action)
        if edge is None:
            return self._matrix, 0, True, False, info

        i, j = edge

        self._matrix[i,j] += 1 #do
        self._matrix[j,i] += 1 #do
        self._matrix[i,j] %= 2 #undo
        self._matrix[j,i] %= 2 #undo
    
            
        reward = self._reward()
        
        truncated = False

        observation = self._matrix
        return observation, reward, terminated, truncated, info

    def render(self):
        return f"{self._matrix}"
    


### Training the single-agent diameter-maximiser

The environment is wrapped with `ActionMasker`. The model used is a `MaskablePPO` and the masking function returns the vector with all ones as we want all possible moves to be legal. Note that for some more complex problems we will to return a vector with some zero entries to prevent some moves.

In [ ]:
def mask_fn(env):
    return env._legal_actions()

n_verts=4

env = GraphWorldWithUndo(n_verts)
env = ActionMasker(env, mask_fn) 

model = MaskablePPO(
    "MlpPolicy",
    env,
    verbose=0,
    learning_rate=0.0001, #3e-4,
    clip_range=2,
    gamma=0.99,
)

# %%

# %%

model.learn(total_timesteps=500)

# %%
env.env.current_player = 1
obs, _ = env.reset()      
env.env.partner = model



# training loop
total_iterations=20
for i in range(total_iterations):
    
    percent = (i + 1) / total_iterations * 100
    bar_length = 40
    filled = int(bar_length * (i + 1) // total_iterations)
    bar = '█' * filled + '-' * (bar_length - filled)

    print(f"\rProgress: [{bar}] {percent:.1f}%", end="", flush=True)

    # Training
    model.learn(total_timesteps=200)

print("\nTraining completed!")

    

### Visualisiation

The final adjacency matrix is converted to a `networkx` graph and the graph is shown.

In [ ]:
# reset env
obs, info = env.reset()
terminated = False

while not terminated:
    action, _ = model.predict(obs, action_masks=info["action_mask"])
    obs, reward, terminated, truncated, info = env.step(action)

adj_matrix = obs[:, :env.env.nverts]
G = nx.from_numpy_array(adj_matrix)

plt.figure(figsize=(8, 6))
pos = nx.spring_layout(G)
nx.draw(G, pos, with_labels=True, node_color='lightblue',
        node_size=500, font_size=16, font_weight='bold',
        edge_color='gray', width=2)
plt.show()

# Questions and Experiments for Reinforcement Learning on Graph Diameter Prediction

## Single-Agent PPO

### Understanding the Learning Process

1. How does the performance change as the number of vertices increases?

2. How does the PPO clipping parameter affect training stability and convergence?

3. What is the effect of increasing the entropy bonus during training?

4. How does reward shaping influence the learning process?
---

 # Regular Graphs

### `RegularGraphWorld` (single-player variant)

This class extends `GraphWorldWithUndo` with a different objective: instead of maximising diameter, the agent tries to build a **$k$-regular graph**, fixing the number of vertices and regularity.

**Reward design**  
The reward is based on `delta_k` the change in the *count* of vertices whose degree equals exactly $k$. A positive delta means more vertices have reached the target degree. The step penalty of `−0.5` discourages the agent from making unnecessary moves.

As an exercise, try to built the class `RegularGraphWorld(GraphWorldWithUndo)` and train the agent to build `k`-regular graphs, giving as inputs the regularity `k` and the number of vertices `nverts`.

In [ ]:

class RegularGraphWorld(GraphWorldWithUndo):
    def __init__(self, nverts, k): 
        ...

### Training on the regularity objective (single-player)

Same structure as before: create a `MaskablePPO` agent, and run the training loop.

In [ ]:
def mask_fn(env):
    return env._legal_actions()

n_verts=8
k=3 #regularity

env = RegularGraphWorld(n_verts,k) # (nverts, k)
env = ActionMasker(env, mask_fn) 

model = MaskablePPO(
    "MlpPolicy",
    env,
    verbose=0,
    ent_coef=0.02,
    learning_rate=0.0001,   #3e-4,
    clip_range=0.2, #2,
    gamma=0.99,
)

# %%

# %%

model.learn(total_timesteps=500)

# %%
env.env.current_player = 1
obs, _ = env.reset()      
env.env.partner = model



# training loop

total_iterations = 100
for i in range(total_iterations):
    
    percent = (i + 1) / total_iterations * 100
    bar_length = 40
    filled = int(bar_length * (i + 1) // total_iterations)
    bar = '█' * filled + '-' * (bar_length - filled)

    print(f"\rProgress: [{bar}] {percent:.1f}%", end="", flush=True)

    # Training
    model.learn(total_timesteps=200)

print("\nTraining completed!")

    

### Visualising the regular graph

After rollout, the degree sequence is extracted and compared against the target $k$. We also display the min/max degree. A perfectly $k$-regular graph has min = max = $k$. The graph is displayed.

As a final challenge, train the agent to construct the Petersen graph, a well-known `(10,3)`-regular graph.

In [ ]:
# reset env
obs, _ = env.reset()
terminated = False

while not terminated:
    action, _ = model.predict(obs)
    obs, reward, terminated, truncated, _ = env.step(action)

# adjacency matrix (no +1 column anymore)
adj_matrix = obs

G = nx.from_numpy_array(adj_matrix)

plt.figure(figsize=(8, 6))
pos = nx.spring_layout(G)

nx.draw(
    G,
    pos,
    with_labels=True,
    node_color='lightblue',
    node_size=500,
    font_size=16,
    font_weight='bold',
    edge_color='gray',
    width=2
)

deg = [G.degree(v) for v in G.nodes()]

plt.title("Generated graph")
plt.show()

print("Degree sequence:", deg)
print("Min degree:", min(deg))
print("Max degree:", max(deg))

# Questions and Experiments for the (k)-Regular Graph Construction Game

## Single-Agent PPO

1. How does the learning process change as the number of vertices increases?

2. How does the task difficulty change as the regularity $k$ increases?

3. Are some values of $n$ and $k$ harder than others?

---


# Two players - game

# Maximise the diameter

### Redesigned `GraphWorldWithUndo` — two-player cooperative protocol

The notebook now pivots to a **two-agent cooperative design**. Edge selection is split across two agents using a message-passing protocol encoded directly in the adjacency matrix:

**How the protocol works**  
The observation matrix is extended from `nverts × nverts` to `nverts × (nverts+1)`. The extra column acts as a *message channel*:

1. **Player 0** picks a vertex $i$ → writes a message into column `nverts` via `_send_msg(i)`.
2. **Player 1** reads that message via `_get_msg()` and picks a second vertex $j ≠ i$ → the edge $(i, j)$ is toggled.

This cooperative split means each player only has to choose *one endpoint* of an edge rather than the full edge, reducing the effective action space from $O(n^2)$ to $O(n)$.

At any step the two players can decide to resign and return the current graph as the solution.

**Action masking**  
- Player 0's mask: all vertices are legal plus the resign action.
- Player 1's mask: all vertices except the one nominated by Player 0 are legal (see the method `_legal_actions_p1`). If no message exists, only resign is available.

**Subclasses**  
- `MaxDiameterEnv`: rewards the *change* in graph diameter (stripped of the message column before computing).
- `RegularGraphWorld`: rewards movement of the degree sequence towards $k$-regularity using the L2-norm sign.

In [ ]:
# Our base model
class GraphWorldWithUndo(gym.Env):
    def __init__(self, nverts):
        self.nverts = nverts
    
        # Adjacency matrix with column describing move
        self.observation_space = gym.spaces.Box(
            0, 1, shape=(nverts, nverts + 1), dtype=np.int8
        )

        self._n_actions = nverts + 1
        self._steps = 0

        self.action_space = gym.spaces.Discrete(self._n_actions)
        self.render_mode = "ansi"
        self.current_player = 0
        self.partner = None
        self.previous_invariant = 0
        self.reset()
        
    # We view the rightmost "move" column as communicating a message
    def _clear_msg(self):
        self._matrix[:, -1] = np.zeros(self.nverts)
        
    def _send_msg(self, i):
        self._clear_msg()
        self._matrix[i, -1] = 1
        
    def _get_msg(self):
        n = np.argmax(self._matrix[:, -1])
        if self._matrix[n, -1]:
            return n

    def _legal_actions_p0(self):
        return np.ones(self._n_actions, dtype=np.bool_)

    def _legal_actions_p1(self):
        m = self._get_msg()
        mask = np.zeros(self._n_actions, dtype=np.bool_)
        mask[-1] = True
        if m is None:
            return mask
        for i in range(self.nverts):
            mask[i] =  i != m
        return mask
    
    # To be specified!
    def _reward(self):
        return 0
       
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self._matrix = np.zeros((self.nverts,self.nverts+1),dtype=np.int8)
        self.previous_invariant = self._reward()
        self._steps = 0
        
        if self.current_player == 0:
            info = {"action_mask": self._legal_actions_p0()}
            return self._matrix, info
        else:
            move0 = self.nverts
            while move0 ==  self.nverts:
                move0, _ = self.partner.predict(self._matrix)
            self._send_msg(move0)
            info = {"action_mask": self._legal_actions_p1()}
            return self._matrix, info

    def step(self, action):
        self._steps += 1
        # Resignation
        if action == self.nverts:
            terminated = True
            return self._matrix, 0, terminated, False, {}
        # Otherwise we play :)
        if self.current_player == 0:
            self._send_msg(action)
            action2, _ = self.partner.predict(self._matrix,action_masks=self._legal_actions_p1())
            terminated = False
            if action == self.nverts or action2 == self.nverts:
                terminated = True
                return self._matrix, 0, terminated, False, {}
            
            self._matrix[action, action2] += 1
            self._matrix[action2, action] += 1
            self._matrix[action, action2] %= 2
            self._matrix[action2, action] %= 2
            self._clear_msg()
            info = {"action_mask": self._legal_actions_p0()}
            
        if self.current_player == 1:
            action1 = self._get_msg()
            if action1 is None: 
                terminated = True
                return self._matrix, 0, terminated, False, {}
            
            self._clear_msg()
            self._matrix[action1, action] += 1
            self._matrix[action, action1] += 1
            self._matrix[action1, action] %= 2
            self._matrix[action, action1] %= 2
            
            action2, _ = self.partner.predict(self._matrix,action_masks=self._legal_actions_p0())
            terminated = False
            if action2 == self.nverts:
                terminated = True
            else:
                self._send_msg(action2)
            info = {"action_mask": self._legal_actions_p1()}
            
        reward = self._reward()
        
        truncated = False
        
        observation = self._matrix
        return observation, reward, terminated, truncated, info

    def render(self):
        return f"{self._matrix}"
    
# Models that inherit ---------------------------------------------------------

class MaxDiameterEnv(GraphWorldWithUndo):
    def _diameter(self, graph):
        return max(nx.diameter(nx.subgraph(graph, comp)) for comp in nx.connected_components(graph))
    
    def _compute_reward(self, current_invariant, previous_invariant):
        return current_invariant - previous_invariant

    def _reward(self):
        adj_mat = self._matrix[:,:-1]
        G = nx.from_numpy_array(adj_mat)
        new_invariant = self._diameter(G)
        reward = self._compute_reward(new_invariant, self.previous_invariant)
        self.previous_invariant = new_invariant
        return reward

    
def masked(env):
    from sb3_contrib.common.wrappers import ActionMasker

    def mask_fn(env) -> np.ndarray:
        if env.current_player == 0:
            return env._legal_actions_p0()
        if env.current_player == 1:
            return env._legal_actions_p1()
    return ActionMasker(env, mask_fn)

def create_models(env, verbose=0, learning_rate=0.0001, clip_range=2, gamma=0.99):
    from sb3_contrib import MaskablePPO
    return tuple(MaskablePPO("MlpPolicy",
                        env,
                        verbose=verbose,
                        learning_rate=learning_rate,
                        clip_range=clip_range,
                        gamma=gamma) for _ in [0,1])

### Model initialisation

Two separate `MaskablePPO` instances (`model0`, `model1`) are created, one per player.

### Training loop - diameter

The two agents are trained in **alternating self-play**: each iteration, `model0` trains as Player 0 with `model1` as its fixed partner, then `model1` trains as Player 1 with `model0` as its fixed partner. The partners are updated every iteration, so each agent continuously adapts to an improving collaborator.

In [ ]:
# =========================================================
# ACTION MASK
# =========================================================

def mask_fn(env) -> np.ndarray:
    # must return a boolean array of shape (n_actions,)
    if env.current_player == 0:
        return env._legal_actions_p0()   
    if env.current_player == 1:
        return env._legal_actions_p1()
    

# =========================================================
# ENV
# =========================================================

n_verts = 4

base_env = MaxDiameterEnv(n_verts)
env = ActionMasker(base_env, mask_fn)

# =========================================================
# MODELS
# =========================================================

learning_rate = 3e-4
clip_range = 0.2
gamma = 0.999
entropy_coefficient = 0.05
rollout_steps = 1024

model0 = MaskablePPO(
    "MlpPolicy",
    env,
    verbose=0,
    learning_rate=learning_rate,
    clip_range=clip_range,
    gamma=gamma,
    ent_coef=entropy_coefficient,
    n_steps=rollout_steps,
)

model1 = MaskablePPO(
    "MlpPolicy",
    env,
    verbose=0,
    learning_rate=learning_rate,
    clip_range=clip_range,
    gamma=gamma,
    ent_coef=entropy_coefficient,
    n_steps=rollout_steps,
)

print("Models created!")

# =========================================================
# INITIAL PARTNERS
# =========================================================

env.env.current_player = 0
env.env.partner = model1

# =========================================================
# TRAINING LOOP
# =========================================================

total_iterations = 50
train_steps = 200

for i in range(total_iterations):

    percent = (i + 1) / total_iterations * 100
    bar_length = 40
    filled = int(bar_length * (i + 1) // total_iterations)
    bar = '█' * filled + '-' * (bar_length - filled)

    print(f"\rProgress: [{bar}] {percent:.1f}%", end="", flush=True)

    # -------------------------------------------------
    # Train model0 as player 0
    # -------------------------------------------------

    env.env.current_player = 0
    env.env.partner = model1

    model0.learn(
        total_timesteps=train_steps,
        reset_num_timesteps=False
    )

    # -------------------------------------------------
    # Train model1 as player 1
    # -------------------------------------------------

    env.env.current_player = 1
    env.env.partner = model0

    model1.learn(
        total_timesteps=train_steps,
        reset_num_timesteps=False
    )

print("\nTraining completed!")

### Evaluating the diameter agent

The resulting graph and its diameter are displayed.

In [ ]:
# =========================================================
# TEST MODEL
# =========================================================

# choose who you want to test
test_model = model0

# the partner has to be the other model
env.env.partner = model1

# choose player
env.env.current_player = 0

# reset env
obs, info = env.reset()

terminated = False
total_reward = 0

while not terminated:

    action, _ = test_model.predict(
        obs,
        action_masks=info["action_mask"],
        deterministic=True
    )

    obs, reward, terminated, truncated, info = env.step(action)

    total_reward += reward

print("Total reward:", total_reward)

# =========================================================
# BUILD GRAPH
# =========================================================

adj_matrix = obs[:, :env.env.nverts]

G = nx.from_numpy_array(adj_matrix)

print("Diameter:", nx.diameter(G) if nx.is_connected(G) else "Disconnected")

# =========================================================
# DRAW GRAPH
# =========================================================

plt.figure(figsize=(8, 6))

pos = nx.spring_layout(G, seed=42)

nx.draw(
    G,
    pos,
    with_labels=True,
    node_color='lightblue',
    node_size=700,
    font_size=14,
    font_weight='bold',
    edge_color='gray',
    width=2
)

plt.title("Generated Graph")

plt.show()


# Two-Player PPO

1. How does the performance change as the number of vertices increases?

2. Do cooperative agents learn faster than a single agent?

3. Is the training process more stable or less stable than in the single-agent setting?

4. How does communication or shared information affect performance?
---

# Regularity

The same two-player architecture is now applied to the **regularity objective** using `RegularGraphWorld`. The environment, reward, and action-masking logic are identical to those described above; only the target changes from diameter to degree regularity.

As an exercise, try to construct the `RegularGraphWorld(GraphWorldWithUndo)`, adapted to the two-player game.

In [ ]:
class RegularGraphWorld(GraphWorldWithUndo):
    def __init__(self, nverts, k): 
        ...

In [ ]:
# Alternative class without "undo" and with masking moves that increase degrees above k
class RegularGraphWorld(gym.Env):
    def __init__(self, nverts,k):
        self.nverts = nverts
    
        # Adjacency matrix with column describing move
        self.observation_space = gym.spaces.Box(
            0, 1, shape=(nverts, nverts + 1), dtype=np.int8
        )

        self._n_actions = nverts
        self._steps = 0
        self.k = k

        self.action_space = gym.spaces.Discrete(self._n_actions)
        self.render_mode = "ansi"
        self.current_player = 0
        self.partner = None
        self.previous_invariant = 0
        self.starting_deg_seq = np.zeros((2,self.nverts))
        self.deg_seqs = np.concat((self.starting_deg_seq, self.starting_deg_seq), axis=0)
        self.reset()
        
    # We view the rightmost "move" column as communicating a message
    def _clear_msg(self):
        self._matrix[:, -1] = np.zeros(self.nverts)
        
    def _send_msg(self, i):
        self._clear_msg()
        self._matrix[i, -1] = 1
        
    def _get_msg(self):
        n = np.argmax(self._matrix[:, -1])
        if self._matrix[n, -1]:
            return n

    def _legal_actions_p0(self):
        deg = self.deg_seqs[1]
        # P1 mask: can nominate any vertex still needing edges
        p1_valid = deg < self.k
        p1_mask = p1_valid
        if not p1_mask.any():
            p1_mask = np.ones(self._n_actions, dtype=np.bool_)  # terminal fallback
        return p1_mask

    def _legal_actions_p1(self):
        deg = self.deg_seqs[1]
        nominated = self._get_msg()
        if nominated is not None:
            p2_valid = deg < self.k
            p2_valid[nominated] = False              # no self-loop
            # no duplicate edge
            p2_valid[self._matrix[nominated, :self.nverts].astype(bool)] = False
        else:
            p2_valid = deg < self.k                  # fallback if no message yet
        
        
        if not p2_valid.any():
            p2_valid = np.ones(self.nverts, dtype=np.bool_)
        
        p2_mask = p2_valid
        return p2_mask
    
    # To be specified!
    def _reward(self):
        return 0
       
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self._matrix = np.zeros((self.nverts,self.nverts+1),dtype=np.int8)
        self.previous_invariant = self._reward()
        self.deg_seqs = np.zeros((2,self.nverts))
        self._steps = 0
        
        if self.current_player == 0:
            info = {"action_mask": self._legal_actions_p0()}
            return self._matrix, info
        else:
            move0 = self.nverts
            while move0 ==  self.nverts:
                move0, _ = self.partner.predict(self._matrix)
            self._send_msg(move0)
            info = {"action_mask": self._legal_actions_p1()}
            return self._matrix, info
        
    def _check_done(self) -> bool:
        #return (self._step_count >= self.max_steps)
        deg = self.deg_seqs[1]
        # Check directly without relying on mask/message
        #p2_can_move = (deg < self.k).sum() >= 2  # need at least 2 vertices with room
        #return not p2_can_move
        return np.count_nonzero(deg < self.k) < 2

    def step(self, action):
        self._steps += 1
        # Resignation
        if action == self.nverts:
            terminated = True
            return self._matrix, 0, terminated, False, {}
        # Otherwise we play :)
        if self.current_player == 0:
            self._send_msg(action)
            action2, _ = self.partner.predict(self._matrix,action_masks=self._legal_actions_p1())
            terminated = False
            self.deg_seqs[0] = self.deg_seqs[1].copy()
            self.deg_seqs[1][action2] +=1
            self.deg_seqs[1][action] +=1
            if self._check_done():
                terminated = True
                return self._matrix, 0, terminated, False, {}
            
            self._matrix[action, action2] += 1
            self._matrix[action2, action] += 1
            #self._matrix[action, action2] %= 2
            #self._matrix[action2, action] %= 2
            self._clear_msg()
            info = {"action_mask": self._legal_actions_p0()}
            
        if self.current_player == 1:
            action1 = self._get_msg()
            if action1 is None: 
                terminated = True
                return self._matrix, 0, terminated, False, {}
            
            self._clear_msg()
            self._matrix[action1, action] += 1
            self._matrix[action, action1] += 1
            #self._matrix[action1, action] %= 2
            #self._matrix[action, action1] %= 2

            self.deg_seqs[0] = self.deg_seqs[1].copy()
            self.deg_seqs[1][action1] +=1
            self.deg_seqs[1][action] +=1
            
            action2, _ = self.partner.predict(self._matrix,action_masks=self._legal_actions_p0())
            terminated = False
            if action2 == self.nverts:
                terminated = True
            else:
                self._send_msg(action2)
            info = {"action_mask": self._legal_actions_p1()}
            
        reward = self._reward()

        terminated = self._check_done()
        
        truncated = False
        
        observation = self._matrix
        return observation, reward, terminated, truncated, info

    def render(self):
        return f"{self._matrix}"
    

In [ ]:
# =========================================================
# ACTION MASK
# =========================================================

def mask_fn(env):
    if env.current_player == 0:
        return env._legal_actions_p0()
    else:
        return env._legal_actions_p1()

# =========================================================
# PARAMETERS
# =========================================================

LOAD = False

n_verts = 8
k_reg = 3

learning_rate = 3e-4
clip_range = 0.2
gamma = 0.9999
entropy_coefficient = 0.1
rollout_steps = 1024




# =========================================================
# ENVIRONMENT
# =========================================================

base_env = RegularGraphWorld(n_verts, k_reg)

env = ActionMasker(base_env, mask_fn)

# =========================================================
# MODELS
# =========================================================

model0 = MaskablePPO(
    "MlpPolicy",
    env,
    verbose=0,
    learning_rate=learning_rate,
    clip_range=clip_range,
    gamma=gamma,
    ent_coef=entropy_coefficient,
    n_steps=rollout_steps,
)

model1 = MaskablePPO(
    "MlpPolicy",
    env,
    verbose=0,
    learning_rate=learning_rate,
    clip_range=clip_range,
    gamma=gamma,
    ent_coef=entropy_coefficient,
    n_steps=rollout_steps,
)

print("Models created!")


### Training loop - regularity

Same alternating self-play structure. Here the training is more demanding because the reward landscape is sparser: the agent only gets a signal when the degree sequence moves strictly closer to or further from the target $k$-regular sequence.

In [ ]:

# =========================================================
# TRAINING LOOP
# =========================================================
total_iterations = 30


for i in range(total_iterations):

    percent = (i + 1) / total_iterations * 100

    bar_length = 40
    filled = int(bar_length * (i + 1) // total_iterations)

    bar = '█' * filled + '-' * (bar_length - filled)

    print(f"\rProgress: [{bar}] {percent:.1f}%", end="", flush=True)

    # -----------------------------------------------------
    # Train model0 as player 0
    # -----------------------------------------------------

    env.env.current_player = 0
    env.env.partner = model1

    model0.learn(
        total_timesteps=rollout_steps,
        reset_num_timesteps=False
    )

    # -----------------------------------------------------
    # Train model1 as player 1
    # -----------------------------------------------------

    env.env.current_player = 1
    env.env.partner = model0

    model1.learn(
        total_timesteps=rollout_steps,
        reset_num_timesteps=False
    )

print("\nTraining completed!")


In [ ]:

# =========================================================
# TEST MODEL
# =========================================================

test_model = model0

env.env.current_player = 0
env.env.partner = model1

obs, info = env.reset()

terminated = False
total_reward = 0

while not terminated:

    action, _ = test_model.predict(
        obs,
        action_masks=info["action_mask"],
        deterministic=True
    )

    obs, reward, terminated, truncated, info = env.step(action)

    total_reward += reward

print("\nTotal reward:", total_reward)


### Analysing the resulting graph

We display a graph whose degree sequence is printed alongside the plot, making it easy to check how close the agent came to $k$-regularity.

In [ ]:
# =========================================================
# GRAPH ANALYSIS
# =========================================================

adj_matrix = obs[:, :env.env.nverts]

G = nx.from_numpy_array(adj_matrix)

degrees = np.array([d for _, d in G.degree()])

print("Degree sequence:", degrees)

print("Target degree:", k_reg)

print("Min degree:", degrees.min())
print("Max degree:", degrees.max())

# =========================================================
# DRAW GRAPH
# =========================================================

plt.figure(figsize=(8, 6))

pos = nx.spring_layout(G, seed=42)

nx.draw(
    G,
    pos,
    with_labels=True,
    node_color='lightblue',
    node_size=700,
    font_size=14,
    font_weight='bold',
    edge_color='gray',
    width=2
)

plt.title(f"{k_reg}-Regularity Environment")

plt.show()

## Two-Player PPO

1. How does the learning process change as the number of vertices increases?

2. How does the task difficulty change as the degree $k$ increases?

3. Are some values of $n$ and $k$ easier than others?

---


# Comparing Single-Agent and Multi-Agent PPO

## Diameter

1. Which setting converges faster?

2. Which approach requires more iterations to be trained?

3. Does the multi-agent setting require more exploration?

---


## Regularity

1. Which setting learns valid $k$-regular graphs faster?

2. Does the two agent game require more exploration?

4. Which setting is more stable during training?

5. Which approach solves better the task to construct the Petersen Graph?

---